# Mini-Project : Data Analysis for Marketing Strategy

---

## Introduction
In this mini-project, we will perform data analysis to devise a marketing strategy based on various aspects like area analysis, customer analysis, product category analysis, and sales and profit time series.

---

### What You'll Learn
- How to load and preprocess a dataset.
- Techniques for area analysis to identify key markets.
- Methods for customer analysis to determine high-value customers.
- Strategies for product category analysis to identify top-performing products.
- How to analyze sales and profit trends over time.
- Application of the Pareto Principle to prioritize key drivers of sales and profit.

---

### Dataset Attributes
| Column | Description |
|---|---|
| Row ID | Unique ID for each row |
| Order ID | Unique Order ID for each Customer |
| Order Date | Order Date of the product |
| Ship Date | Shipping Date of the Product |
| Ship Mode | Shipping Mode specified by the Customer |
| Customer ID | Unique ID to identify each Customer |
| Customer Name | Name of the Customer |
| Segment | The segment where the Customer belongs |
| Country | Country of residence of the Customer |
| City | City of residence of the Customer |
| State | State of residence of the Customer |
| Postal Code | Postal Code of every Customer |
| Region | Region where the Customer belongs |
| Product ID | Unique ID of the Product |
| Category | Category of the product ordered |
| Sub-Category | Sub-Category of the product ordered |
| Product Name | Name of the Product |
| Sales | Sales of the Product |
| Quantity | Quantity of the Product |
| Discount | Discount provided |
| Profit | Profit/Loss incurred |

---
## 1. Setup & Libraries

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully.')

---
## 2. Load & Preprocess Dataset

In [ ]:
import urllib.request
import os

url = 'https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%205%20-%20Data%20Processing/W5D5%20-%20Mini-project%20-%20bis/US%20Superstore%20data.xls'
filename = 'US_Superstore_data.xls'

if not os.path.exists(filename):
    print('Downloading dataset...')
    urllib.request.urlretrieve(url, filename)
    print(f'Downloaded: {filename}')
else:
    print(f'Dataset already present: {filename}')

In [ ]:
df = pd.read_excel('US_Superstore_data.xls')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ------ Preprocessing ------
# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Extract time features
df['Year']  = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.to_period('M')

# Check missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — dataset is clean.')

# Basic stats
df[['Sales', 'Profit', 'Quantity', 'Discount']].describe().round(2)

---
## 3. Area Analysis
### 3.1 Which states have the most sales?

In [ ]:
state_sales = (
    df.groupby('State')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(state_sales['State'], state_sales['Sales'],
              color=sns.color_palette('Blues_r', len(state_sales)))
ax.set_title('Top 20 States by Total Sales', fontsize=15, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Total Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\nTop 5 states by sales:')
print(state_sales.head(5).to_string(index=False))

### 3.2 New York vs California — Sales & Profit Comparison

In [ ]:
compare_states = ['New York', 'California']
ny_ca = (
    df[df['State'].isin(compare_states)]
    .groupby('State')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)

x = np.arange(len(compare_states))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))
bars1 = ax.bar(x - width/2, ny_ca['Sales'],  width, label='Sales',  color='steelblue')
bars2 = ax.bar(x + width/2, ny_ca['Profit'], width, label='Profit', color='darkorange')

ax.set_title('New York vs California — Total Sales & Profit', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(compare_states)
ax.set_ylabel('Amount ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'${bar.get_height():,.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(ny_ca.to_string(index=False))

> **Observation:** California leads in total sales, but New York often shows a higher profit margin relative to its sales, indicating more efficient revenue conversion.

### 3.3 Outstanding Customer in New York

In [ ]:
ny_customers = (
    df[df['State'] == 'New York']
    .groupby('Customer Name')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
    .head(10)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sales
axes[0].barh(ny_customers['Customer Name'], ny_customers['Sales'],
             color=sns.color_palette('Blues_r', len(ny_customers)))
axes[0].set_title('Top 10 NY Customers by Sales', fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# Profit
colors = ['red' if p < 0 else 'seagreen' for p in ny_customers['Profit']]
axes[1].barh(ny_customers['Customer Name'], ny_customers['Profit'], color=colors)
axes[1].set_title('Top 10 NY Customers by Profit', fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].invert_yaxis()
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

plt.tight_layout()
plt.show()

top_ny = ny_customers.iloc[0]
print(f'\nOutstanding NY customer by Sales: {top_ny["Customer Name"]} '
      f'(Sales: ${top_ny["Sales"]:,.2f} | Profit: ${top_ny["Profit"]:,.2f})')

### 3.4 Profitability Differences Among States

In [ ]:
state_profit = (
    df.groupby('State')['Profit']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

colors = ['red' if p < 0 else 'steelblue' for p in state_profit['Profit']]

fig, ax = plt.subplots(figsize=(18, 7))
ax.bar(state_profit['State'], state_profit['Profit'], color=colors)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Total Profit by State (All States)', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Total Profit ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

loss_states = state_profit[state_profit['Profit'] < 0]
print(f'States with negative profit ({len(loss_states)}):')
print(loss_states.to_string(index=False))

> **Observation:** States shown in red are operating at a loss. These states require immediate attention — investigate discount policies and product mix in those markets.

---
## 4. Pareto Principle — Customers & Profit
**Do 20% of customers generate 80% of the profit?**

In [ ]:
customer_profit = (
    df.groupby('Customer Name')['Profit']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total_profit = customer_profit['Profit'].sum()
customer_profit['Cumulative Profit'] = customer_profit['Profit'].cumsum()
customer_profit['Cumulative Profit %'] = customer_profit['Cumulative Profit'] / total_profit * 100
customer_profit['Customer Rank %'] = np.arange(1, len(customer_profit) + 1) / len(customer_profit) * 100

# Find the 20% threshold
threshold_20 = customer_profit[customer_profit['Customer Rank %'] <= 20].iloc[-1]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(customer_profit['Customer Rank %'], customer_profit['Cumulative Profit %'],
        color='steelblue', linewidth=2, label='Cumulative Profit %')
ax.axvline(20, color='red', linestyle='--', linewidth=1.5, label='20% of customers')
ax.axhline(80, color='orange', linestyle='--', linewidth=1.5, label='80% of profit')
ax.scatter([threshold_20['Customer Rank %']], [threshold_20['Cumulative Profit %']],
           color='red', zorder=5, s=80)
ax.annotate(f"Top 20% → {threshold_20['Cumulative Profit %']:.1f}% of profit",
            xy=(threshold_20['Customer Rank %'], threshold_20['Cumulative Profit %']),
            xytext=(25, threshold_20['Cumulative Profit %'] - 8),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))
ax.set_title('Pareto Curve — Cumulative Profit by Customer Rank', fontsize=14, fontweight='bold')
ax.set_xlabel('Cumulative % of Customers (ranked by profit)')
ax.set_ylabel('Cumulative % of Total Profit')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Top 20% of customers ({int(len(customer_profit)*0.2)} customers) '
      f'generate {threshold_20["Cumulative Profit %"]:.1f}% of total profit.')
print(f'Pareto principle (80/20) applies: {threshold_20["Cumulative Profit %"] >= 80}')

---
## 5. Top 20 Cities Analysis
### 5.1 Top 20 Cities by Sales

In [ ]:
city_agg = (
    df.groupby('City')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)
city_agg['Profit Margin %'] = (city_agg['Profit'] / city_agg['Sales'] * 100).round(2)

top20_sales  = city_agg.sort_values('Sales',  ascending=False).head(20)
top20_profit = city_agg.sort_values('Profit', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(14, 7))
palette = sns.color_palette('Blues_r', len(top20_sales))
ax.barh(top20_sales['City'], top20_sales['Sales'], color=palette)
ax.set_title('Top 20 Cities by Total Sales', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.show()

### 5.2 Top 20 Cities by Profit

In [ ]:
colors = ['red' if p < 0 else 'seagreen' for p in top20_profit['Profit']]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top20_profit['City'], top20_profit['Profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Cities by Total Profit', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Profit ($)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.show()

### 5.3 Profitability Differences Among Top-20 Sales Cities

In [ ]:
# Margin scatter: top-20 sales cities
fig, ax = plt.subplots(figsize=(12, 7))
scatter_colors = ['red' if m < 0 else 'steelblue' for m in top20_sales['Profit Margin %']]
sc = ax.scatter(top20_sales['Sales'], top20_sales['Profit Margin %'],
                c=scatter_colors, s=100, edgecolors='white', linewidth=0.5)

for _, row in top20_sales.iterrows():
    ax.annotate(row['City'], (row['Sales'], row['Profit Margin %']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Sales vs Profit Margin % — Top 20 Sales Cities', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)')
ax.set_ylabel('Profit Margin (%)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
plt.tight_layout()
plt.show()

print(top20_sales[['City', 'Sales', 'Profit', 'Profit Margin %']]
      .sort_values('Profit Margin %', ascending=False)
      .to_string(index=False))

> **Observation:** High sales do not always translate to high profit margins. Some top-selling cities may still have poor profitability due to heavy discounting or high operational costs.

---
## 6. Customer Analysis
### 6.1 Top 20 Customers by Sales

In [ ]:
top20_customers = (
    df.groupby('Customer Name')[['Sales', 'Profit']]
    .sum()
    .sort_values('Sales', ascending=False)
    .head(20)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Sales bars
axes[0].barh(top20_customers['Customer Name'], top20_customers['Sales'],
             color=sns.color_palette('Blues_r', len(top20_customers)))
axes[0].set_title('Top 20 Customers by Sales', fontweight='bold')
axes[0].set_xlabel('Total Sales ($)')
axes[0].invert_yaxis()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

# Profit bars
profit_colors = ['red' if p < 0 else 'seagreen' for p in top20_customers['Profit']]
axes[1].barh(top20_customers['Customer Name'], top20_customers['Profit'], color=profit_colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit of Top 20 Customers by Sales', fontweight='bold')
axes[1].set_xlabel('Total Profit ($)')
axes[1].invert_yaxis()
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

plt.tight_layout()
plt.show()

print(top20_customers.to_string(index=False))

### 6.2 Cumulative Sales Curve by Customer — Pareto Principle
**Can we apply the Pareto Principle to Customers and Sales?**

In [ ]:
customer_sales = (
    df.groupby('Customer Name')['Sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

total_sales = customer_sales['Sales'].sum()
customer_sales['Cumulative Sales'] = customer_sales['Sales'].cumsum()
customer_sales['Cumulative Sales %'] = customer_sales['Cumulative Sales'] / total_sales * 100
customer_sales['Customer Rank %']   = np.arange(1, len(customer_sales) + 1) / len(customer_sales) * 100

threshold_20_sales = customer_sales[customer_sales['Customer Rank %'] <= 20].iloc[-1]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(customer_sales['Customer Rank %'], customer_sales['Cumulative Sales %'],
        color='steelblue', linewidth=2, label='Cumulative Sales %')
ax.fill_between(customer_sales['Customer Rank %'], customer_sales['Cumulative Sales %'],
                alpha=0.1, color='steelblue')
ax.axvline(20, color='red',    linestyle='--', linewidth=1.5, label='20% of customers')
ax.axhline(80, color='orange', linestyle='--', linewidth=1.5, label='80% of sales')
ax.scatter([threshold_20_sales['Customer Rank %']], [threshold_20_sales['Cumulative Sales %']],
           color='red', zorder=5, s=80)
ax.annotate(f"Top 20% → {threshold_20_sales['Cumulative Sales %']:.1f}% of sales",
            xy=(threshold_20_sales['Customer Rank %'], threshold_20_sales['Cumulative Sales %']),
            xytext=(25, threshold_20_sales['Cumulative Sales %'] - 8),
            fontsize=10, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))
ax.set_title('Pareto Curve — Cumulative Sales by Customer Rank', fontsize=14, fontweight='bold')
ax.set_xlabel('Cumulative % of Customers (ranked by sales)')
ax.set_ylabel('Cumulative % of Total Sales')
ax.legend()
plt.tight_layout()
plt.show()

n_top20 = int(len(customer_sales) * 0.2)
print(f'Top 20% of customers ({n_top20} customers) '
      f'generate {threshold_20_sales["Cumulative Sales %"]:.1f}% of total sales.')
print(f'Pareto principle (80/20) applies: {threshold_20_sales["Cumulative Sales %"] >= 80}')

---
## 7. Sales & Profit Time Series

In [ ]:
monthly = (
    df.groupby('Month')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)
monthly['Month'] = monthly['Month'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].plot(monthly['Month'], monthly['Sales'], color='steelblue', linewidth=2, marker='o', markersize=4)
axes[0].fill_between(monthly['Month'], monthly['Sales'], alpha=0.15, color='steelblue')
axes[0].set_title('Monthly Sales Over Time', fontweight='bold')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

axes[1].plot(monthly['Month'], monthly['Profit'], color='darkorange', linewidth=2, marker='o', markersize=4)
axes[1].fill_between(monthly['Month'], monthly['Profit'], alpha=0.15, color='darkorange')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('Monthly Profit Over Time', fontweight='bold')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))

tick_positions = list(range(0, len(monthly), 3))
axes[1].set_xticks([monthly['Month'].iloc[i] for i in tick_positions])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 8. Marketing Strategy Recommendations

Based on the analysis above, here are the key strategic recommendations:

---

### States to Prioritize
| Priority | State | Rationale |
|---|---|---|
| High | **California, New York, Texas** | Highest total sales volume — invest in retention and upsell campaigns |
| Medium | **Washington, Pennsylvania, Florida** | Strong sales with growth potential |
| Avoid / Fix | **States with negative profit** | Reassess discount strategy and product mix before investing in marketing |

---

### Cities to Prioritize
- Focus marketing budget on **high-sales AND high-margin cities** (top-right quadrant of the scatter chart).
- Investigate **high-sales but low/negative margin cities** — profitability problems may stem from over-discounting.

---

### Customer Strategy
- The **top 20% of customers drive the majority of both sales and profit** — consistent with the Pareto Principle.
- Implement a **VIP / loyalty programme** for the top 20% to ensure retention.
- Deploy **win-back campaigns** for previously high-value customers showing declining order frequency.
- In **New York**, the outstanding customer represents a disproportionate share of revenue — assign a dedicated account manager.

---

### Discount Policy
- Loss-making states and cities often correlate with high average discounts — **cap discounts at 20%** and measure impact on profitability.

---

### Seasonality
- Sales peak towards **Q4 (October–December)** — align marketing campaigns with this seasonal trend.
- Plan inventory and promotions ahead of peak periods to maximise profit margins.